# different ship headings experiment

question: how can a 15° change in starting heading change whether a ship hits a Seattle waterfront building? uses the same 60 m ship from the same location at rest in the saved 5 m initial-surge simulation on a 12 m GeoClaw grid

In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from netCDF4 import Dataset
from shapely import wkt
from shapely.geometry import box
from shapely.ops import unary_union, transform
sys.path.insert(0, str(Path('../../src').resolve()))
from geoclaw_debris.ship_tracking import Flow, Ship, simulate, hull_polygon

ds = Dataset('pike_place_surge5_fgout.nc')
assert ds.terrain_sha256 == hashlib.sha256(Path('seattle_buildings_mhw.tt3').read_bytes()).hexdigest()
assert ds.surge_height_m == 5.0 and ds.grid_spacing_m == 12.0
flow = Flow(ds)
extent = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
buildings = unary_union([p for p in (wkt.loads(s) for s in pd.read_csv('../../SeattleBuildings/Seattle_Inventory.csv').geometry) if p.intersects(extent)])

In [ ]:
ship = Ship()
start = (-122.3425, 47.6055)
headings = range(0,180,15)
tracks = {}
for heading in headings:
    track = pd.DataFrame(simulate(flow, buildings, ship, start, heading_deg=heading, contact_mode='stop'))
    stop = track.index[track.status != 'afloat']
    tracks[heading] = track if stop.empty else track.loc[:stop[0]]
ds.close()

results = pd.DataFrame({heading: track.iloc[-1] for heading, track in tracks.items()}).T
hit = results.status == 'building contact'
print(f'{hit.sum()} of {len(headings)} headings make building contact')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
colors = {'building contact': 'firebrick', 'afloat': 'teal'}
for status, rows in results.groupby('status'):
    ax.scatter(rows.index, rows.time_s, color=colors.get(status, 'gray'), label=status)
ax.set(xticks=list(headings), xlabel='starting heading (° from east)', ylabel='time (s)')
ax.legend()
fig.tight_layout()

In [ ]:
origin = np.array(flow.to_xy(*start))
xlim, ylim = (-40,110), (-65,75)
view = box(origin[0]+xlim[0], origin[1]+ylim[0], origin[0]+xlim[1], origin[1]+ylim[1])
building_xy = transform(flow.to_xy, buildings)
polys = [p for p in building_xy.geoms if p.intersects(view)]
fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharex=True, sharey=True)
examples = [15,30,60]
trails, hulls = [], []
for ax, heading in zip(axes, examples):
    track = tracks[heading]
    last = track.iloc[-1]
    color = colors.get(last.status, 'gray')
    for poly in polys:
        ax.add_patch(PolygonPatch(np.asarray(poly.exterior.coords)-origin, facecolor='lightgray', edgecolor='gray', lw=0.7))
    trail, = ax.plot(track.x_m-origin[0], track.y_m-origin[1], color=color, lw=2)
    trails.append(trail)
    for row, ls in [(track.iloc[0], '--'), (last, '-')]:
        outline = hull_polygon((row.x_m,row.y_m), np.deg2rad(row.heading_deg), ship)
        ax.add_patch(PolygonPatch(np.asarray(outline.exterior.coords)-origin, fill=False, edgecolor=color, ls=ls))
    hulls.append(ax.patches[-1])
    ax.set(xlim=xlim, ylim=ylim, xlabel='east of release (m)', title=f'{heading}°: {last.status} at {last.time_s:.0f} s')
    ax.set_aspect('equal')
axes[0].set_ylabel('north of release (m)')
fig.tight_layout()
fig.subplots_adjust(top=0.82)
plt.show()

## Animation

All three releases share the same clock. After contact, the hull stays at its last position.

In [ ]:
def update(t):
    for ax, heading, trail, hull in zip(axes, examples, trails, hulls):
        track = tracks[heading].loc[:t]
        row = track.iloc[-1]
        outline = hull_polygon((row.x_m,row.y_m), np.deg2rad(row.heading_deg), ship)
        hull.set_xy(np.asarray(outline.exterior.coords)-origin)
        trail.set_data(track.x_m-origin[0], track.y_m-origin[1])
        status = row.status if row.status == 'afloat' else f'{row.status} at {row.time_s:.0f} s'
        ax.set_title(f'{heading}°: {status}')
    fig.suptitle(f'{t} s')

frames = sorted(set(range(0,601,10)) | {int(tracks[h].time_s.iloc[-1]) for h in examples})
anim = FuncAnimation(fig, update, frames=frames, interval=120)
plt.close(fig)
HTML(anim.to_jshtml())